# HanuAi ML Assessment — Task 2: Advanced EDA and Text Mining
EDA, rule-based entity extraction, issue-type categorization, and TF-IDF/KMeans clustering over warranty-claim data.

In [1]:
import re
import json
import warnings
from collections import Counter

import pandas as pd
import numpy as np

warnings.filterwarnings("ignore")

INPUT_CSV = "../data/task2_raw.csv"
OUTPUT_CSV = "../output/task2_tagged.csv"
OUTPUT_SUMMARY_JSON = "../output/task2_insights_summary.json"


# ---------------------------------------------------------------------------
# 1. LOAD + EDA
# ---------------------------------------------------------------------------

In [2]:
def load_and_profile(path: str = INPUT_CSV) -> pd.DataFrame:
    df = pd.read_csv(path, parse_dates=["Opened date", "BUILD_DATE", "IN_USE_DATE"])
    print(f"Loaded {len(df)} rows, {len(df.columns)} columns from {path}\n")

    print("=" * 70)
    print("COLUMN PROFILE")
    print("=" * 70)
    profile = pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "n_missing": df.isna().sum(),
        "pct_missing": (100 * df.isna().mean()).round(1),
        "n_unique": df.nunique(),
    })
    print(profile.to_string())

    n_dupe_ids = df["Event id"].duplicated().sum()
    n_dupe_rows = df.duplicated().sum()
    print(f"\nDuplicate Event IDs: {n_dupe_ids}")
    print(f"Fully duplicate rows: {n_dupe_rows}")

    # Age-to-fail: days between build and the service event being opened.
    # A classic reliability-engineering signal stakeholders care about.
    df["days_build_to_open"] = (df["Opened date"] - df["BUILD_DATE"]).dt.days
    df["days_inuse_to_open"] = (df["Opened date"] - df["IN_USE_DATE"]).dt.days

    print("\nBuild-to-claim lag (days) — describe():")
    print(df["days_build_to_open"].describe().round(1).to_string())

    # CAUSAL_CD_DESC has a handful of truly blank/whitespace-only values
    # beyond pandas' NaN count (seen as '' in the raw export) — normalize.
    for col in ["CAUSAL_CD_DESC"]:
        df[col] = df[col].replace(r"^\s*$", np.nan, regex=True)

    print(f"\nMAKE values: {df['MAKE'].unique().tolist()}")
    print(f"MODEL values: {df['MODEL'].nunique()} unique")
    print(f"Model years: {sorted(df['MODLYR'].dropna().unique().tolist())}")
    print(f"Plants: {df['PLANT'].unique().tolist()}")

    return df


# ---------------------------------------------------------------------------
# 2. TEXT MINING — rule-based entity extraction
# ---------------------------------------------------------------------------
# Built by reading the vocabulary used in the 5 gold rows (Failure Component,
# Failure Condition, Trigger, Fix Component, Fix Condition) and extending
# coverage against CAUSAL_CD_DESC / COMPLAINT_CD_DESC so untagged rows map
# onto the same label space the gold rows already use, rather than
# inventing a parallel taxonomy.

COMPONENT_KEYWORDS = {
    "Radio": [r"\bradio\b"],
    "Display": [r"\bdisplay\b", r"\bscreen\b"],
    "Audio Unit": [r"\baudio unit\b", r"\bamplifier\b", r"\bamp\b", r"\bspeaker"],
    "Radio Module": [r"\bradio module\b"],
    "OnStar": [r"\bonstar\b"],
    "Antenna": [r"\bantenna\b"],
    "Bluetooth Module": [r"\bbluetooth\b"],
    "USB Port": [r"\busb\b"],
    "Navigation Module": [r"\bnav(igation)?\b"],
    "Backup Camera": [r"\bback\s*up camera\b", r"\brear camera\b", r"\breverse camera\b"],
    "Microphone": [r"\bmic(rophone)?\b"],
    "Steering Wheel Controls": [r"\bsteering wheel control", r"\bswc\b"],
    "TCICM": [r"\btcicm\b", r"telematics communication"],
    "Wiring Harness": [r"\bwiring harness\b", r"\bwire harness\b", r"\bharness\b"],
    "Coaxial Cable": [r"\bcoax(ial)?\s*cable\b"],
}

CONDITION_KEYWORDS = {
    "Black Screen": [r"\bblack screen\b", r"\bscreen (went|turns?|is) black\b", r"\bblank screen\b"],
    "No Sound": [r"\bno sound\b", r"\bno audio\b"],
    "Inoperative": [r"\binop(erative)?\b", r"\bnot working\b", r"\bdoes(n'| )?t work\b"],
    "Malfunction": [r"\bmalfunction", r"\bfault(y|ing)?\b"],
    "Internal Fault": [r"\binternal (fault|failure)\b"],
    "Controls Irresponsive": [r"\b(button|control|touch)s? (not responding|unresponsive|irresponsive)\b",
                                r"\bquit working\b"],
    "Lost Communication": [r"\blost communication\b", r"\bno/?incorrect communication\b", r"\bcommunication loss\b"],
    "Shorted": [r"\bshort(ed|ing)?\b"],
    "Freezing / Locking Up": [r"\bfreez(e|ing|es)\b", r"\block(s|ed|ing)? up\b"],
    "Resetting / Rebooting": [r"\bre-?(boot|set)(s|ting)?\b", r"\brestart(s|ing)?\b"],
    "Static / Noise": [r"\bstatic\b", r"\bbuzz(ing)?\b", r"\bhum(ming)?\b", r"\bpopping\b", r"\bcrackl"],
    "Connectivity Failure": [r"\bconnection fail", r"\bwon'?t connect\b", r"\bdisconnect(s|ing|ed)?\b",
                               r"\bdrops? connection\b"],
    "Signal Lost": [r"\bsignal lost\b", r"\blost sign", r"\bno signal\b", r"\breception\b"],
    "Software/Programming Error": [r"\bsoftware (update|error)\b", r"\bprogramming error\b",
                                     r"\bsps ?error\b", r"\bcode\b.*\bnot (found|duplicat)"],
    "Cuts Out / Intermittent": [r"\bcut(s)? out\b", r"\bintermitt"],
}

TRIGGER_KEYWORDS = {
    "While Driving": [r"\bwhile driving\b", r"\bdriving down\b", r"\bduring (a )?drive\b"],
    "When Reversing": [r"\breverse\b", r"\breversing\b", r"\bbackup\b", r"\bputting (it |the car )?in reverse\b"],
    "On Startup": [r"\bstart(s|ing)? (the )?(car|vehicle|engine)\b", r"\bupon start\b", r"\bturn(s|ing|ed)? on\b",
                     r"\bcold start\b"],
    "Intermittent / Random": [r"\brandom(ly)?\b", r"\bsometimes\b", r"\bintermitt"],
    "After Software Update": [r"\bafter (the )?(software )?update\b", r"\bafter update\b"],
    "Connecting Phone/Bluetooth": [r"\bconnect(ing)? (a |the |my )?phone\b", r"\bpairing\b", r"\bbluetooth\b"],
    "No Additional Functionality": [],  # fallback label used by the gold rows
}

FIX_COMPONENT_KEYWORDS = {
    "Radio": [r"\breplaced?.{0,15}radio\b", r"\breplace(d)? the radio\b", r"\bnew radio\b", r"\bradio replace"],
    "SPS": [r"\bsps\b"],
    "USB": [r"\busb (file transfer|update|programming)\b"],
    "Antenna": [r"\breplace(d)?.{0,15}antenna\b"],
    "Display": [r"\breplace(d)?.{0,15}display\b"],
    "Wiring Harness": [r"\breplace(d)?.{0,15}harness\b"],
}

FIX_CONDITION_KEYWORDS = {
    "Replaced": [r"\breplace(d|ment)?\b"],
    "Programmed": [r"\bprogram(med|ming)?\b"],
    "Reprogrammed": [r"\breprogram"],
    "Reset": [r"\breset\b"],
    "Updated": [r"\bupdate(d)?\b"],
    "Repaired": [r"\brepair(ed)?\b"],
    "No Trouble Found": [r"\bno trouble found\b", r"\bunable to duplicate\b", r"\bcould not (be )?duplicat"],
}

In [3]:
def _match_tags(text: str, keyword_map: dict, fallback) -> list:
    """
    fallback: a string to use when nothing matches, OR None to return an
    empty list (used where "no hit" is a legitimate, non-noisy outcome
    rather than a forced guess — e.g. failure component, where defaulting
    everything to "Radio" would drown out the real signal).
    """
    if not isinstance(text, str) or not text.strip():
        return [fallback] if fallback else []
    text_l = text.lower()
    hits = []
    for label, patterns in keyword_map.items():
        if not patterns:
            continue
        if any(re.search(p, text_l) for p in patterns):
            hits.append(label)
    if hits:
        return hits
    return [fallback] if fallback else ["Unspecified"]

In [4]:
def extract_entities(row: pd.Series) -> pd.Series:
    """
    Mines each of the 5 structured entity fields from the free-text columns,
    using the same field a human filled the 5 gold rows from (CUSTOMER_VERBATIM
    for the customer-facing trigger/condition, CAUSAL_VERBATIM for root cause,
    CORRECTION_VERBATIM for the fix).
    """
    customer_txt = str(row.get("CUSTOMER_VERBATIM", "") or "")
    causal_txt = str(row.get("CAUSAL_VERBATIM", "") or "")
    correction_txt = str(row.get("CORRECTION_VERBATIM", "") or "")
    combined_diag_txt = causal_txt + " " + customer_txt

    trigger = _match_tags(customer_txt, TRIGGER_KEYWORDS, "No Additional Functionality")
    # Component tagging: no forced "Radio" guess — untagged rows legitimately
    # mean the verbatim didn't name a specific part, which is itself a
    # useful "Unspecified" signal rather than inflated Radio counts.
    failure_component = _match_tags(combined_diag_txt, COMPONENT_KEYWORDS, None)
    failure_condition = _match_tags(combined_diag_txt, CONDITION_KEYWORDS, None)
    additional_context = _match_tags(customer_txt, TRIGGER_KEYWORDS, "No Additional Context")
    # "Additional Context" in the gold rows reuses trigger-like values
    # ("While Driving") or "No Additional Context" when none apply.
    if additional_context == ["No Additional Functionality"]:
        additional_context = ["No Additional Context"]

    fix_component = _match_tags(correction_txt, FIX_COMPONENT_KEYWORDS, None)
    fix_condition = _match_tags(correction_txt, FIX_CONDITION_KEYWORDS, None)

    return pd.Series({
        "mined_trigger": trigger,
        "mined_failure_component": failure_component,
        "mined_failure_condition": failure_condition,
        "mined_additional_context": additional_context,
        "mined_fix_component": fix_component,
        "mined_fix_condition": fix_condition,
    })


ISSUE_TYPE_RULES = [
    # (issue_type, any CAUSAL_CD_DESC substring OR failure_condition tag that implies it)
    ("Communication / Network Failure", ["no/incorrect communication", "lost communication", "connectivity failure"]),
    ("Electrical / Short Circuit", ["shorted", "discharge", "poor contact"]),
    ("Component Failure (Hardware)", ["broken", "damaged/cracked", "worn/stripped", "internal fault"]),
    ("Software / Programming Issue", ["registers incorrectly", "software/programming error", "incorrect version"]),
    ("Alignment / Mechanical", ["alignment/adjustment", "out of balance", "loose", "seized/binds/sticks"]),
    ("No Fault Found / Follow-up", ["no trouble found", "not duplicated", "follow operation", "campaign/ tech bulletin"]),
]

In [5]:
def categorize_issue_type(row: pd.Series) -> str:
    causal_cd = str(row.get("CAUSAL_CD_DESC", "") or "").lower()
    cond_tags = " ".join(row.get("mined_failure_condition", [])).lower()
    hay = causal_cd + " " + cond_tags
    for issue_type, needles in ISSUE_TYPE_RULES:
        if any(n in hay for n in needles):
            return issue_type
    return "Other / Unclassified"


# ---------------------------------------------------------------------------
# 3. UNSUPERVISED CROSS-CHECK — TF-IDF + KMeans clustering
# ---------------------------------------------------------------------------

In [6]:
def cluster_complaints(df: pd.DataFrame, n_clusters: int = 8) -> pd.DataFrame:
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.cluster import KMeans

    texts = df["CUSTOMER_VERBATIM"].fillna("").astype(str)

    # Boilerplate report-writing phrases ("customer states", "check and
    # advise") appear in nearly every verbatim regardless of the actual
    # failure, so they dominate TF-IDF term weight and wash out real
    # signal unless explicitly suppressed.
    boilerplate_stopwords = [
        "customer", "customers", "cust", "states", "state", "stated",
        "check", "advise", "advised", "concern", "concerns", "vehicle",
        "complaint", "complains", "complaining", "reports", "reported",
    ]
    stop_words = list(TfidfVectorizer(stop_words="english").get_stop_words()) + boilerplate_stopwords

    vectorizer = TfidfVectorizer(
        max_features=500, stop_words=stop_words, ngram_range=(1, 2), min_df=3
    )
    X = vectorizer.fit_transform(texts)

    km = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    labels = km.fit_predict(X)
    df["cluster_id"] = labels

    # Name each cluster by its top TF-IDF terms (cheap, explainable "topic").
    terms = vectorizer.get_feature_names_out()
    cluster_names = {}
    for c in range(n_clusters):
        center = km.cluster_centers_[c]
        top_idx = center.argsort()[::-1][:5]
        cluster_names[c] = ", ".join(terms[i] for i in top_idx)
    df["cluster_topic"] = df["cluster_id"].map(cluster_names)

    print("\nCluster topics (top TF-IDF terms):")
    for c, name in cluster_names.items():
        n = (df["cluster_id"] == c).sum()
        print(f"  Cluster {c} (n={n}): {name}")

    return df


# ---------------------------------------------------------------------------
# 4. INSIGHTS SUMMARY
# ---------------------------------------------------------------------------

In [7]:
def generate_insights(df: pd.DataFrame) -> dict:
    insights = {}

    insights["total_events"] = int(len(df))
    insights["top_failure_components"] = Counter(
        tag for tags in df["mined_failure_component"] for tag in tags
    ).most_common(10)
    insights["top_failure_conditions"] = Counter(
        tag for tags in df["mined_failure_condition"] for tag in tags
    ).most_common(10)
    insights["issue_type_distribution"] = df["issue_type"].value_counts().to_dict()
    insights["top_models_by_event_count"] = df["MODEL"].value_counts().head(10).to_dict()
    insights["top_plants_by_event_count"] = df["PLANT"].value_counts().head(10).to_dict()
    insights["avg_days_build_to_claim"] = round(float(df["days_build_to_open"].mean()), 1)
    insights["fix_condition_distribution"] = Counter(
        tag for tags in df["mined_fix_condition"] for tag in tags
    ).most_common(10)

    # Model x issue-type cross-tab — flags which models skew toward which
    # failure type (useful for a build-quality / supplier root-cause review).
    cross = pd.crosstab(df["MODEL"], df["issue_type"])
    insights["model_issue_crosstab_top_rows"] = cross.loc[
        df["MODEL"].value_counts().head(5).index
    ].to_dict(orient="index")

    return insights


# ---------------------------------------------------------------------------
# MAIN
# ---------------------------------------------------------------------------

In [8]:
def run():
    df = load_and_profile()

    print("\n" + "=" * 70)
    print("TEXT MINING — entity extraction")
    print("=" * 70)
    mined = df.apply(extract_entities, axis=1)
    df = pd.concat([df, mined], axis=1)

    df["issue_type"] = df.apply(categorize_issue_type, axis=1)

    print("\nIssue type distribution:")
    print(df["issue_type"].value_counts().to_string())

    print("\n" + "=" * 70)
    print("CLUSTERING — TF-IDF + KMeans over CUSTOMER_VERBATIM")
    print("=" * 70)
    df = cluster_complaints(df, n_clusters=8)

    # Serialize list-valued columns to strings so the CSV stays flat/Excel-safe.
    list_cols = [c for c in df.columns if c.startswith("mined_")]
    df_out = df.copy()
    for c in list_cols:
        df_out[c] = df_out[c].apply(lambda v: json.dumps(v))

    df_out.to_csv(OUTPUT_CSV, index=False)
    print(f"\nSaved tagged output to {OUTPUT_CSV}")

    insights = generate_insights(df)
    with open(OUTPUT_SUMMARY_JSON, "w") as f:
        json.dump(insights, f, indent=2, default=str)
    print(f"Saved insights summary to {OUTPUT_SUMMARY_JSON}")

    print("\n" + "=" * 70)
    print("TOP-LINE INSIGHTS")
    print("=" * 70)
    print("Top failure components:", insights["top_failure_components"][:5])
    print("Top failure conditions:", insights["top_failure_conditions"][:5])
    print("Issue type split:", insights["issue_type_distribution"])

    return df

In [9]:
if __name__ == "__main__":
    df = run()

Loaded 1000 rows, 20 columns from ../data/task2_raw.csv

COLUMN PROFILE
                              dtype  n_missing  pct_missing  n_unique
Event id                        str          0          0.0      1000
Opened date          datetime64[us]          0          0.0       397
CAUSAL_VERBATIM                 str          1          0.1       944
CORRECTION_VERBATIM             str          0          0.0       952
CUSTOMER_VERBATIM               str          0          0.0       990
BUILD_DATE           datetime64[us]          0          0.0        37
BUILD_PLANT_DESC                str          0          0.0         8
CAUSAL_CD_DESC                  str          3          0.3        38
COMPLAINT_CD_DESC               str          0          0.0         9
IN_USE_DATE          datetime64[us]          1          0.1       285
MAKE                            str          0          0.0         4
MODEL                           str          0          0.0        23
MODLYR            


Issue type distribution:
issue_type
Communication / Network Failure    450
Electrical / Short Circuit         275
Software / Programming Issue       112
Component Failure (Hardware)        87
No Fault Found / Follow-up          53
Alignment / Mechanical              13
Other / Unclassified                10

CLUSTERING — TF-IDF + KMeans over CUSTOMER_VERBATIM



Cluster topics (top TF-IDF terms):
  Cluster 0 (n=59): working, radio working, camera working, camera, radio
  Cluster 1 (n=328): display, camera, screen, work, phone
  Cluster 2 (n=67): inop, radio inop, camera inop, radio, camera
  Cluster 3 (n=63): goes, screen goes, goes blank, blank, screen
  Cluster 4 (n=83): xm, xm radio, radio, signal, play
  Cluster 5 (n=264): radio, shut, sop, times, stays
  Cluster 6 (n=110): screen, radio screen, radio, screen black, black
  Cluster 7 (n=26): card, sd, sd card, removed, card removed

Saved tagged output to ../output/task2_tagged.csv
Saved insights summary to ../output/task2_insights_summary.json

TOP-LINE INSIGHTS
Top failure components: [('Radio', 925), ('Display', 415), ('Backup Camera', 97), ('USB Port', 63), ('Navigation Module', 61)]
Top failure conditions: [('Unspecified', 334), ('Inoperative', 215), ('Malfunction', 195), ('Internal Fault', 150), ('Cuts Out / Intermittent', 117)]
Issue type split: {'Communication / Network Failure': 